In [ ]:
from dotenv import load_dotenv

load_dotenv()

Model initialisation

In [1]:
from langchain.chat_models import init_chat_model
model= init_chat_model(model="gpt-5-mini")

Create MCP SERVERS and agents


In [4]:
from typing import Dict, Any
from langchain.tools import tool
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def search_web(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

client = MultiServerMCPClient({
    "kiwi": {
        "transport": "streamable_http",
        "url": "https://mcp.kiwi.com",
    }
})

flight_tools = await client.get_tools()
print([t.name for t in flight_tools])

flight_agent = create_agent(
    model=model,
    tools=flight_tools,
    system_prompt="You are a travel agent. Find the cheapest flights for the request. Give the airline, price, date and duration.",
)

['search-flight', 'feedback-to-devs']


In [6]:
venue_agent = create_agent(
    model=model,
    tools=[search_web],
    system_prompt="You are a wedding venue expert. Search the web and suggest 3 wedding venues that match the city, number of guests and budget. Give the name, location and approximate price.",
)

In [7]:
playlist_agent = create_agent(
    model=model,
    tools=[search_web],
    system_prompt="You are a wedding DJ. Create a playlist of 10 songs based on the music styles the user likes. Give the title and the artist for each song.",
)

In [9]:
from langchain.messages import HumanMessage

@tool
async def find_flights(request: str) -> str:
    """Find flights for the wedding. Give the departure city, destination, dates and number of passengers."""
    result = await flight_agent.ainvoke({"messages": [HumanMessage(content=request)]})
    return result["messages"][-1].content

@tool
async def find_venue(request: str) -> str:
    """Find a wedding venue. Give the city, number of guests and budget."""
    result = await venue_agent.ainvoke({"messages": [HumanMessage(content=request)]})
    return result["messages"][-1].content

@tool
async def make_playlist(request: str) -> str:
    """Create a wedding playlist. Give the music styles or artists the user likes."""
    result = await playlist_agent.ainvoke({"messages": [HumanMessage(content=request)]})
    return result["messages"][-1].content

In [10]:
from langgraph.checkpoint.memory import InMemorySaver

wedding_planner = create_agent(
    model=model,
    tools=[find_flights, find_venue, make_playlist],
    checkpointer=InMemorySaver(),
    system_prompt="""You are a wedding planner. You help the user with 3 things:
1. Flights: use find_flights.
2. The wedding venue: use find_venue.
3. A music playlist: use make_playlist.

If information is missing (cities, dates, number of guests, budget, music tastes), ask the user before using the tools.
At the end, give one clear plan with the flights, the venue and the playlist.""",
)

config = {"configurable": {"thread_id": "1"}}

In [12]:
question = HumanMessage(content=(
    "We are getting married in Rome on 2026-12-20. "
    "2 of us fly from Paris. We will have 80 guests and a budget of 15,000 euros for the venue.don't ask me question  "
    "We love jazz and soul music."
))

response = await wedding_planner.ainvoke({"messages": [question]}, config)
print(response["messages"][-1].content)

Thanks — I’ve searched flights, shortlisted venues, and created an initial jazz/soul playlist for your wedding in Rome on 2026-12-20. I’ll book or finalize anything only if you tell me to proceed.

Here’s a single clear plan combining the flights, the venue shortlist, and the playlist:

Flights (suggested)
- Option to book: Wizz Air Malta — ORY → FCO departing 19 Dec 2026 at 09:55 arriving 12:00; return FCO → ORY 22 Dec 2026 dep 07:10 arr 09:20. Total ~€382 for 2 in economy (personal item only). Morning departures available. Ask if you want checked bags or CDG-only search.

Venue (recommended shortlist to contact/confirm)
- Antico Arco (Trastevere) — indoor private rooms, fits ~80–95, estimated €10k–€15k total.
- Hotel Ponte Sisto — central hotel, private function rooms, estimated €8k–€14k total.
- Palazzo Montemartini — 5-star hotel near Termini, modern salons, estimated €12k–€18k total (may exceed the top end depending on choices).

Music (jazz & soul playlist — highlights with first